In [28]:
!pip3 install -q langchain-google-genai google-generativeai langchain-community langchain-core langchain_classic requests duckduckgo-search ddgs langchain_experimental

In [38]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
import requests

In [39]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

In [40]:
# Step 1: Create tool
@tool

def get_weather_data(city: str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key={{WeatherStack API KEY}}&query={city}'

  response = requests.get(url)

  return response.json()

In [41]:
API_KEY = "AQ###################"

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=API_KEY
)

In [42]:
# Step 2: Creating prompt

from langchain_core.prompts import PromptTemplate

template = '''Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}'''

prompt = PromptTemplate.from_template(template)

In [43]:
from langchain_classic.agents import create_react_agent, AgentExecutor

# Step 3: Creating agent
agent = create_react_agent(
    llm=llm,
    tools=[search_tool, get_weather_data],
    prompt=prompt
)

In [44]:
# Step 4: Wraping agent with AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather_data],
    verbose=True
)

In [45]:
# Step 5: Invoke
response = agent_executor.invoke({"input": "Find the capital of Madhya Pradesh, then find it's current weather condition"})
print(response)



> Entering new AgentExecutor chain...
I need to find the capital of Madhya Pradesh first. Then, I will use the `get_weather_data` tool to find the current weather condition of that capital city.
Action: duckduckgo_search
Action Input: capital of Madhya PradeshAfter the independence of India, Madhya Pradesh was created in 1950 from the former British Central Provinces and Berar and the princely states of Makrai and Chhattisgarh, with Nagpur as the capital of the state. The capital of Madhya Pradesh, Bhopal, is a city of administrative power, cultural richness, and natural beauty. Its central location, historical importance, and well-developed infrastructure make it ideal for governance and development. Madhya Pradesh, state of India. As its name implies—madhya means “central” and pradesh means “region” or “state”—it is situated in the heart of the country. The state has no coastline and no international frontier. It is bounded by the states of Uttar Pradesh to the northeast, Chhattisg

In [46]:
response['output']

'The capital of Madhya Pradesh is Bhopal. The current weather condition in Bhopal is partly cloudy with a temperature of 36°C.'